# Ready-made AI services (Azure AI services, formerly "Cognitive Services")

Not every problem needs your own model. For standard tasks – *what's the sentiment of this review*, *translate this*, *what's in this photo*, *read the text from this document* – the cloud providers offer **deep learning models already trained on huge datasets**, behind a simple API. That's the **SaaS / model-as-a-service** level from the class notes.

Under the hood it's exactly what we learned:
- **Language** and **Translator** are Transformer models (Class 3),
- **Vision** uses convolutional networks and Vision Transformers (Class 4),
- **Speech** turns audio into text and back with sequence models.

All of them live in one resource of type *Azure AI services* – the same resource that hosts our language model for Class 6. You pay per call (per 1 000 text records, per 1 000 images…), and there is a free tier for experimenting.

## Setup

Same as for the language model: the **endpoint** of the resource (Azure portal → the AI services / Foundry resource → *Overview*) and, in Google Colab, your **tenant (directory) ID** for the device-code login. Your account needs the role *Cognitive Services User* on the resource – no API keys in code.

In [ ]:
%pip install -q requests azure-identity matplotlib pillow

In [ ]:
import os
import sys
import requests
from azure.identity import AzureCliCredential, DeviceCodeCredential

ENDPOINT = (os.environ.get("AZURE_AI_SERVICES_ENDPOINT") or os.environ.get("AZURE_OPENAI_ENDPOINT")
            or input("AI services endpoint (https://....cognitiveservices.azure.com/): ")).strip().rstrip("/")

if "google.colab" in sys.modules:
    tenant = os.environ.get("AZURE_TENANT_ID") or input("Tenant (directory) ID: ").strip()
    credential = DeviceCodeCredential(tenant_id=tenant)   # prints a link + code to log in
else:
    credential = AzureCliCredential()                     # reuses `az login`


def call(path, params, body):
    """POST to an Azure AI service with an Entra ID token and return the JSON answer."""
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    r = requests.post(f"{ENDPOINT}/{path}", params=params, json=body,
                      headers={"Authorization": f"Bearer {token}"}, timeout=60)
    if not r.ok:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()

## 1. Language: sentiment analysis

Is a text positive, negative or neutral? Remember the IMDB sentiment classifier we trained in Class 3 on 25 000 reviews? This one is ready to use, works in Polish, and also scores every sentence separately.

In [ ]:
def sentiment(texts, language="pl"):
    body = {"kind": "SentimentAnalysis",
            "analysisInput": {"documents": [{"id": str(i), "language": language, "text": t}
                                            for i, t in enumerate(texts)]},
            "parameters": {"opinionMining": True}}
    result = call("language/:analyze-text", {"api-version": "2023-04-01"}, body)
    for doc, text in zip(result["results"]["documents"], texts):
        s = doc["confidenceScores"]
        print(f"{doc['sentiment']:>8}  (+{s['positive']:.2f} / ={s['neutral']:.2f} / -{s['negative']:.2f})  {text}")


sentiment([
    "Zajęcia z deep learningu były świetne!",
    "Projekt był za trudny i nikt nam nie pomógł.",
    "Zajęcia z deep learningu były świetne, ale sala była za zimna.",
    "Wykład zaczyna się o 17:00.",
])

Look at the third sentence: positive about the class, negative about the room. One label for the whole text hides that. **Opinion mining** (switched on above) goes further and links each opinion to its target – that's called *aspect-based* sentiment:

In [ ]:
body = {"kind": "SentimentAnalysis",
        "analysisInput": {"documents": [{"id": "1", "language": "en",
                                         "text": "The lectures were great, but the room was too cold and the coffee was awful."}]},
        "parameters": {"opinionMining": True}}
doc = call("language/:analyze-text", {"api-version": "2023-04-01"}, body)["results"]["documents"][0]
for sentence in doc["sentences"]:
    for target in sentence.get("targets", []):
        print(f"{target['text']:>10} → {target['sentiment']}")

## 2. Language: named entities and key phrases

**Named entity recognition (NER)** finds people, organizations, places, dates, amounts… – a token-level task, exactly what BERT was designed for (Class 3).

In [ ]:
text = ("Pamela Krzypkowska prowadzi zajęcia z deep learningu na Akademii Leona Koźmińskiego w Warszawie "
        "w poniedziałki o 17:00, a projekty trzeba oddać do 7 grudnia.")

result = call("language/:analyze-text", {"api-version": "2023-04-01"},
              {"kind": "EntityRecognition",
               "analysisInput": {"documents": [{"id": "1", "language": "pl", "text": text}]}})
for e in result["results"]["documents"][0]["entities"]:
    print(f"{e['category']:<15} {e['text']:<40} {e['confidenceScore']:.2f}")

In [ ]:
result = call("language/:analyze-text", {"api-version": "2023-04-01"},
              {"kind": "KeyPhraseExtraction",
               "analysisInput": {"documents": [{"id": "1", "language": "pl", "text": text}]}})
print(result["results"]["documents"][0]["keyPhrases"])

## 3. Translator

Neural machine translation – the task the original Transformer paper was written for (Class 3). One call, several target languages, and the source language is detected automatically.

In [ ]:
result = call("translator/text/v3.0/translate", {"api-version": "3.0", "to": ["en", "de", "uk"]},
              [{"Text": "Sieci neuronowe uczą się na danych, a nie na regułach."}])
print("detected:", result[0]["detectedLanguage"])
for t in result[0]["translations"]:
    print(f"{t['to']}: {t['text']}")

## 4. Vision: tags, objects and reading text (OCR)

Image analysis – the descendants of the convnets from Class 4. We ask for three things at once:
- **tags** – what's in the picture (multi-label classification, sigmoid per class!),
- **objects** – what *and where* (object detection, with bounding boxes),
- **read** – printed or handwritten text (OCR).

(Automatic image captions are available only in some Azure regions – not in ours. In Class 6 we'll let the language model describe images instead.)

In [ ]:
import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

IMAGES = {
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "stop sign": "http://images.cocodataset.org/val2017/000000000724.jpg",
}


def analyze(url):
    return call("computervision/imageanalysis:analyze",
                {"api-version": "2024-02-01", "features": "tags,objects,read"}, {"url": url})


for name, url in IMAGES.items():
    result = analyze(url)
    image = Image.open(io.BytesIO(requests.get(url, timeout=30).content))

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(image)
    for obj in result["objectsResult"]["values"]:
        b = obj["boundingBox"]
        ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"], fill=False, color="red", linewidth=2))
        ax.text(b["x"], b["y"] - 5, f"{obj['tags'][0]['name']} {obj['tags'][0]['confidence']:.2f}",
                color="white", backgroundcolor="red", fontsize=9)
    ax.axis("off")
    ax.set_title(name)
    plt.show()

    tags = [f"{t['name']} ({t['confidence']:.2f})" for t in result["tagsResult"]["values"][:8]]
    text = [line["text"] for block in result["readResult"]["blocks"] for line in block["lines"]]
    print("tags:", ", ".join(tags))
    print("text found:", text or "-")

## 5. Ready-made service or your own model?

| Use a ready-made service when… | Train / fine-tune your own model when… |
|---|---|
| The task is standard (sentiment, translation, OCR, generic objects) | The task is specific to your domain (defects on *your* production line, *your* document types) |
| You have little or no labelled data | You have good labelled data |
| You need results today | You need control over the model, its errors and its updates |
| The volume is moderate (you pay per call) | The volume is huge (per-call prices add up) or data can't leave your servers |

In practice companies often start with a ready-made service, collect data and feedback, and only then train their own model where the service isn't good enough.

**Speech** (speech-to-text, text-to-speech with natural voices, also in Polish) is easiest to try in the browser: Azure AI Foundry portal → *Playgrounds* → *Speech*.

## Your turn

1. Find a Polish sentence where the sentiment service is clearly wrong (sarcasm is a good candidate). Why is it hard?
2. Run NER on a news article. Which entity types does it find? Which does it miss?
3. Translate an idiom ("kill two birds with one stone", "nie mój cyrk, nie moje małpy") into three languages. Literal or meaningful?
4. Analyze a photo of a handwritten note. How well does the OCR read it?